In the previous notebook 1modeling_experiments.ipynb, we found that the Prophet model performed well in terms of forecasting rat sightings by day citywide. In this notebook, we will do some more feature engineering and hyperparameter tuning to obtain a better optimal model. Because we wish this to be reusable, we will write things for Prophet and NeuralProphet separately. We first check Prophet's performance and try some new parameters, and then consider NeuralProphet's performance and it's performance as we tune its parameters.

# Import Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging


Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


# Prophet

## Import the data

In [2]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


In [3]:
rs["ds"] = pd.to_datetime(rs["ds"])
full_range = pd.date_range(start="2020-01-01", end="2025-02-28", freq="D")
# reindex and fill missing with 0
rs = rs.set_index("ds").reindex(full_range, fill_value=0).rename_axis("ds").reset_index()
rs

,ds,y
0,2020-01-01,7
1,2020-01-02,21
2,2020-01-03,13
3,2020-01-04,8
4,2020-01-05,5
...,...,...
1881,2025-02-24,20
1882,2025-02-25,24
1883,2025-02-26,28
1884,2025-02-27,31


## Prepare Prophet

In [4]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [5]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [6]:
rs_saved = rs.copy()
df = rs.copy()

## Gridsearch for Hyperparameter Tuning for Prophet

In [7]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [8]:
# This code block is a grid search for hyperparameters.
# To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
# So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

init_days = f'{len(rs)-(14*14)} days'
cv_period = '14 days'
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.1, 1],
    'seasonality_prior_scale': [1,2,5],
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here
performance = []

# Use cross validation to evaluate all parameters
for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)  # Fit model with given params
    df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=14)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

print(best_params)

22:30:02 - cmdstanpy - INFO - Chain [1] start processing
22:30:02 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/13 [00:00<?, ?it/s]

22:30:02 - cmdstanpy - INFO - Chain [1] start processing
22:30:03 - cmdstanpy - INFO - Chain [1] done processing
22:30:03 - cmdstanpy - INFO - Chain [1] start processing
22:30:03 - cmdstanpy - INFO - Chain [1] done processing
22:30:03 - cmdstanpy - INFO - Chain [1] start processing
22:30:03 - cmdstanpy - INFO - Chain [1] done processing
22:30:04 - cmdstanpy - INFO - Chain [1] start processing
22:30:04 - cmdstanpy - INFO - Chain [1] done processing
22:30:04 - cmdstanpy - INFO - Chain [1] start processing
22:30:04 - cmdstanpy - INFO - Chain [1] done processing
22:30:05 - cmdstanpy - INFO - Chain [1] start processing
22:30:05 - cmdstanpy - INFO - Chain [1] done processing
22:30:05 - cmdstanpy - INFO - Chain [1] start processing
22:30:05 - cmdstanpy - INFO - Chain [1] done processing
22:30:05 - cmdstanpy - INFO - Chain [1] start processing
22:30:05 - cmdstanpy - INFO - Chain [1] done processing
22:30:06 - cmdstanpy - INFO - Chain [1] start processing
22:30:06 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/13 [00:00<?, ?it/s]

22:30:08 - cmdstanpy - INFO - Chain [1] start processing
22:30:08 - cmdstanpy - INFO - Chain [1] done processing
22:30:09 - cmdstanpy - INFO - Chain [1] start processing
22:30:09 - cmdstanpy - INFO - Chain [1] done processing
22:30:09 - cmdstanpy - INFO - Chain [1] start processing
22:30:09 - cmdstanpy - INFO - Chain [1] done processing
22:30:09 - cmdstanpy - INFO - Chain [1] start processing
22:30:10 - cmdstanpy - INFO - Chain [1] done processing
22:30:10 - cmdstanpy - INFO - Chain [1] start processing
22:30:10 - cmdstanpy - INFO - Chain [1] done processing
22:30:10 - cmdstanpy - INFO - Chain [1] start processing
22:30:10 - cmdstanpy - INFO - Chain [1] done processing
22:30:11 - cmdstanpy - INFO - Chain [1] start processing
22:30:11 - cmdstanpy - INFO - Chain [1] done processing
22:30:11 - cmdstanpy - INFO - Chain [1] start processing
22:30:11 - cmdstanpy - INFO - Chain [1] done processing
22:30:12 - cmdstanpy - INFO - Chain [1] start processing
22:30:12 - cmdstanpy - INFO - Chain [1]

...

  0%|          | 0/13 [00:00<?, ?it/s]

22:30:30 - cmdstanpy - INFO - Chain [1] start processing
22:30:30 - cmdstanpy - INFO - Chain [1] done processing
22:30:31 - cmdstanpy - INFO - Chain [1] start processing
22:30:31 - cmdstanpy - INFO - Chain [1] done processing
22:30:31 - cmdstanpy - INFO - Chain [1] start processing
22:30:32 - cmdstanpy - INFO - Chain [1] done processing
22:30:32 - cmdstanpy - INFO - Chain [1] start processing
22:30:33 - cmdstanpy - INFO - Chain [1] done processing
22:30:33 - cmdstanpy - INFO - Chain [1] start processing
22:30:33 - cmdstanpy - INFO - Chain [1] done processing
22:30:34 - cmdstanpy - INFO - Chain [1] start processing
22:30:34 - cmdstanpy - INFO - Chain [1] done processing
22:30:34 - cmdstanpy - INFO - Chain [1] start processing
22:30:35 - cmdstanpy - INFO - Chain [1] done processing
22:30:35 - cmdstanpy - INFO - Chain [1] start processing
22:30:35 - cmdstanpy - INFO - Chain [1] done processing
22:30:36 - cmdstanpy - INFO - Chain [1] start processing
22:30:36 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/13 [00:00<?, ?it/s]

22:30:40 - cmdstanpy - INFO - Chain [1] start processing
22:30:40 - cmdstanpy - INFO - Chain [1] done processing
22:30:41 - cmdstanpy - INFO - Chain [1] start processing
22:30:41 - cmdstanpy - INFO - Chain [1] done processing
22:30:41 - cmdstanpy - INFO - Chain [1] start processing
22:30:42 - cmdstanpy - INFO - Chain [1] done processing
22:30:42 - cmdstanpy - INFO - Chain [1] start processing
22:30:42 - cmdstanpy - INFO - Chain [1] done processing
22:30:42 - cmdstanpy - INFO - Chain [1] start processing
22:30:43 - cmdstanpy - INFO - Chain [1] done processing
22:30:43 - cmdstanpy - INFO - Chain [1] start processing
22:30:43 - cmdstanpy - INFO - Chain [1] done processing
22:30:44 - cmdstanpy - INFO - Chain [1] start processing
22:30:44 - cmdstanpy - INFO - Chain [1] done processing
22:30:44 - cmdstanpy - INFO - Chain [1] start processing
22:30:45 - cmdstanpy - INFO - Chain [1] done processing
22:30:45 - cmdstanpy - INFO - Chain [1] start processing
22:30:45 - cmdstanpy - INFO - Chain [1]

{'changepoint_prior_scale': 1, 'seasonality_prior_scale': 2, 'holidays':        holiday         ds  lower_window  upper_window
0   federal_us 2020-01-01             0             1
1   federal_us 2020-01-20             0             1
2   federal_us 2020-02-17             0             1
3   federal_us 2020-05-25             0             1
4   federal_us 2020-07-03             0             1
5   federal_us 2020-09-07             0             1
6   federal_us 2020-10-12             0             1
7   federal_us 2020-11-11             0             1
8   federal_us 2020-11-26             0             1
9   federal_us 2020-12-25             0             1
10  federal_us 2021-01-01             0             1
11  federal_us 2021-01-18             0             1
12  federal_us 2021-02-15             0             1
13  federal_us 2021-05-31             0             1
14  federal_us 2021-06-18             0             1
15  federal_us 2021-07-05             0             1
16  feder

In [9]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(**best_params)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    y_pred = np.round(y_pred)
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

22:30:49 - cmdstanpy - INFO - Chain [1] start processing
22:30:49 - cmdstanpy - INFO - Chain [1] done processing
22:30:50 - cmdstanpy - INFO - Chain [1] start processing
22:30:50 - cmdstanpy - INFO - Chain [1] done processing
22:30:51 - cmdstanpy - INFO - Chain [1] start processing
22:30:51 - cmdstanpy - INFO - Chain [1] done processing
22:30:51 - cmdstanpy - INFO - Chain [1] start processing
22:30:52 - cmdstanpy - INFO - Chain [1] done processing
22:30:52 - cmdstanpy - INFO - Chain [1] start processing
22:30:53 - cmdstanpy - INFO - Chain [1] done processing
22:30:53 - cmdstanpy - INFO - Chain [1] start processing
22:30:54 - cmdstanpy - INFO - Chain [1] done processing
22:30:54 - cmdstanpy - INFO - Chain [1] start processing
22:30:54 - cmdstanpy - INFO - Chain [1] done processing
22:30:55 - cmdstanpy - INFO - Chain [1] start processing
22:30:55 - cmdstanpy - INFO - Chain [1] done processing
22:30:56 - cmdstanpy - INFO - Chain [1] start processing
22:30:56 - cmdstanpy - INFO - Chain [1]

## Prophet's Results

In [10]:
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]
prophet_results_df

,fold,rmse,mape
0,0,7.275203,0.275694
1,1,11.038892,0.596755
2,2,8.040078,0.358834
3,3,4.511889,0.099649
4,4,7.333874,0.122950
5,5,10.049876,0.211894
6,6,7.396911,0.181441
7,7,8.128433,0.259616
8,8,8.228869,0.285086
9,9,6.059231,0.164031


It is possible that the average RMSE displayed above is worse than the one found for Prophet in 1modeling_experiments.ipynb. This would just mean that the gridsearch did not beat the default settings.

In [11]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

# Neural Prophet

## Import the data

In [2]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

rs.head()

,ds,y
1,2020-01-01,7
6,2020-01-02,21
11,2020-01-03,13
16,2020-01-04,8
20,2020-01-05,5


In [3]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [4]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

## Optuna Hyperparameter Tuning

Uncomment the code block below to run Optuna's hyperparameter tuning. Make sure to then comment away the codeblock following that which has a saved result from a previous run.

In [5]:
# we use a different time series split for tuning the hyperparameters to reduce computation time
tscv = TimeSeriesSplit(n_splits=4, test_size=14)

In [6]:
def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 60),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 60),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 7),
    }
    n_lags = trial.suggest_int('n_lags', 1, 60)
    epochs = trial.suggest_int('epochs', 10, 250)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1, log=True)
    batch_size = trial.suggest_int('batch_size', 1024, 2048)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)

        intermediate_score = np.mean(fold_rmses)
        trial.report(intermediate_score, i)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
    return np.mean(fold_rmses)

study = optuna.create_study(direction="minimize")
study = optuna.create_study(
    direction="minimize",
    study_name="model_neural",
    storage="sqlite:///model_neural.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=50)  # adjust n_trials as needed



best_params = study.best_params

print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

[I 2026-03-15 23:05:05,930] A new study created in memory with name: no-name-5dc24329-b8d8-4f12-bcec-e2c489089ef2
[I 2026-03-15 23:05:06,526] Using an existing study with name 'model_neural' instead of creating a new one.
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.945% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.946% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.946% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.946% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


...

Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.946% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.946% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
[I 2026-03-15 23:27:43,909] Trial 51 pruned. 


Best Parameters {'lag_temp_max': 40, 'lag_temp_min': 12, 'lag_snowfall': 2, 'n_lags': 28, 'epochs': 249, 'learning_rate': 0.37721078987363893, 'batch_size': 1320, 'ar_reg': 2.538762924452284}
Best RMSE: 5.2251618137843145


## Evaluate the Model

In [7]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-08
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

wd = pd.DataFrame(data["daily"])
wd["date"] = pd.to_datetime(wd["time"])
wd = wd.set_index("date")

In [8]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]
neural_prophet_results_df

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.934% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


...

Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


,fold,rmse,mape
0,0,6.262863,0.217204
1,1,7.061545,0.342613
2,2,6.649619,0.306428
3,3,5.350225,0.140944
4,4,8.753148,0.165529
5,5,9.976652,0.229369
6,6,8.748305,0.215646
7,7,8.790049,0.268866
8,8,9.014616,0.295128
9,9,7.181031,0.175504


## Results

In [9]:
neural_prophet_results_df

,fold,rmse,mape
0,0,6.262863,0.217204
1,1,7.061545,0.342613
2,2,6.649619,0.306428
3,3,5.350225,0.140944
4,4,8.753148,0.165529
5,5,9.976652,0.229369
6,6,8.748305,0.215646
7,7,8.790049,0.268866
8,8,9.014616,0.295128
9,9,7.181031,0.175504
